# 02 · Preprocesado y feature encoding — Recepción en Steam
**Proyecto ML (Project Break II) · Autor: Isaac Frías**

A partir de las conclusiones del EDA (`01_eda.ipynb`) preparamos los datos para el modelado:

1. Seleccionar las features **sin fuga de información** y separar `X` / `y`.
2. **Split train/test estratificado** (el desbalance 75/25 obliga a estratificar).
3. Definir el **preprocesador** (imputación + `log1p` + escalado + One-Hot) y **ajustarlo solo con
   train**, para no filtrar información del test.

La lógica vive en `src/utils/preprocessing.py`; aquí la aplicamos y la verificamos.

## 0 · Setup y carga

In [1]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils.data_loader import load_raw, clean_steam, add_features, add_target
from src.utils.preprocessing import (
    get_X_y, build_preprocessor, FEATURES, NUM_FEATURES,
    LOG_FEATURES, CAT_FEATURES, BIN_FEATURES, TARGET,
)

CSV = os.environ.get("STEAM_CSV")
dft = add_target(add_features(clean_steam(load_raw(CSV) if CSV else load_raw())))
print(f"Dataset de modelado: {dft.shape[0]:,} juegos")

Dataset de modelado: 30,620 juegos


## 1 · Selección de features y separación X / y

`get_X_y` deja fuera las columnas con fuga (`Positive`, `Negative`, `pct_positivas`, `User score`,
`Recommendations`) y devuelve solo las features utilizables antes/independientemente de las reseñas.

In [2]:
X, y = get_X_y(dft)
print("Features de entrada:", X.shape[1])
print(" · numericas:", NUM_FEATURES)
print(" · categoricas:", CAT_FEATURES)
print(" · binarias:", BIN_FEATURES)
print("\nDistribucion del target:")
print(y.value_counts(normalize=True).round(3).rename("proporcion"))
X.head(3)

Features de entrada: 18
 · numericas: ['Price', 'Required age', 'DLC count', 'Achievements', 'Average playtime forever', 'Peak CCU', 'n_generos', 'n_categorias', 'n_idiomas', 'n_plataformas', 'antiguedad_anios']
 · categoricas: ['genero_principal']
 · binarias: ['es_f2p', 'tiene_publisher', 'tiene_achievements', 'Windows', 'Mac', 'Linux']

Distribucion del target:
recepcion_positiva
1    0.752
0    0.248
Name: proporcion, dtype: float64


,Price,Required age,DLC count,Achievements,Average playtime forever,Peak CCU,n_generos,n_categorias,n_idiomas,n_plataformas,antiguedad_anios,genero_principal,es_f2p,tiene_publisher,tiene_achievements,Windows,Mac,Linux
0,5.24,0,0,0,8,0,1,4,1,1,9,Adventure,0,1,0,1,0,0
1,35.99,0,1,0,675,8,2,4,1,1,0,Simulation,0,1,0,1,0,0
2,0.99,0,1,0,0,0,2,2,2,3,6,Adventure,0,1,0,1,1,1


## 2 · Split train / test (estratificado)

80/20 estratificando por el target para que ambos conjuntos mantengan la proporción 75/25.
`random_state` fijo para reproducibilidad.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)
print(f"train: {X_train.shape[0]:,}  |  test: {X_test.shape[0]:,}")
print("proporcion positiva  train:", round(y_train.mean(), 3),
      "| test:", round(y_test.mean(), 3))

train: 24,496  |  test: 6,124
proporcion positiva  train: 0.752 | test: 0.752


## 3 · Definición del preprocesador

`ColumnTransformer` con cuatro ramas:

| Rama | Columnas | Pasos |
|------|----------|-------|
| log  | Price, playtime, Peak CCU, DLC count, Achievements | imputar mediana → `log1p` → escalar |
| num  | resto de numéricas | imputar mediana → escalar |
| cat  | `genero_principal` | imputar constante → One-Hot (agrupa categorías raras, `min_frequency=50`) |
| bin  | flags (es_f2p, plataformas, …) | passthrough |

El `OneHotEncoder` usa `handle_unknown='ignore'` para no romper si en test aparece un género no visto.

In [4]:
pre = build_preprocessor()
pre

,transformers,"[('log', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## 4 · Ajuste (solo en train) y transformación

**Clave anti-leakage:** `fit_transform` sobre train y `transform` (sin re-fit) sobre test.

In [5]:
X_train_t = pre.fit_transform(X_train)
X_test_t  = pre.transform(X_test)

feat_names = pre.get_feature_names_out()
print(f"Matriz train: {X_train_t.shape}  |  test: {X_test_t.shape}")
print(f"Features tras encoding: {len(feat_names)}  (desde {X.shape[1]} originales)")

Matriz train: (24496, 33)  |  test: (6124, 33)
Features tras encoding: 33  (desde 18 originales)


### 4.1 · Features resultantes

In [6]:
for i, n in enumerate(feat_names):
    print(f"{i:2d}. {n}")

 0. Price
 1. Average playtime forever
 2. Peak CCU
 3. DLC count
 4. Achievements
 5. Required age
 6. n_generos
 7. n_categorias
 8. n_idiomas
 9. n_plataformas
10. antiguedad_anios
11. genero_principal_Action
12. genero_principal_Adventure
13. genero_principal_Animation & Modeling
14. genero_principal_Casual
15. genero_principal_Free To Play
16. genero_principal_Indie
17. genero_principal_Massively Multiplayer
18. genero_principal_RPG
19. genero_principal_Racing
20. genero_principal_Simulation
21. genero_principal_Sin genero
22. genero_principal_Sports
23. genero_principal_Strategy
24. genero_principal_Utilities
25. genero_principal_Violent
26. genero_principal_infrequent_sklearn
27. es_f2p
28. tiene_publisher
29. tiene_achievements
30. Windows
31. Mac
32. Linux


### 4.2 · Comprobaciones de sanidad

Las numéricas escaladas deben quedar con media ≈ 0 y desviación ≈ 1 en train, y no debe quedar
ningún `NaN`.

In [7]:
df_t = pd.DataFrame(X_train_t, columns=feat_names)
chequeo = pd.DataFrame({
    "media": df_t.mean().round(3),
    "std": df_t.std().round(3),
    "nulos": df_t.isna().sum(),
}).head(11)
print(chequeo.to_string())
print("\nNaN totales en train transformado:", int(np.isnan(X_train_t).sum()))
print("NaN totales en test  transformado:", int(np.isnan(X_test_t).sum()))

                          media  std  nulos
Price                      -0.0  1.0      0
Average playtime forever    0.0  1.0      0
Peak CCU                   -0.0  1.0      0
DLC count                   0.0  1.0      0
Achievements                0.0  1.0      0
Required age               -0.0  1.0      0
n_generos                   0.0  1.0      0
n_categorias                0.0  1.0      0
n_idiomas                  -0.0  1.0      0
n_plataformas               0.0  1.0      0
antiguedad_anios            0.0  1.0      0

NaN totales en train transformado: 0
NaN totales en test  transformado: 0


## 5 · Conclusiones y hand-off al modelado

- `X` pasa de **18 features de entrada** a **~33 columnas** tras el One-Hot de `genero_principal`.
- Split **estratificado** 80/20; la proporción 75/25 se mantiene en train y test.
- Preprocesador **ajustado solo con train** → sin fuga hacia el test.
- No quedan nulos y las numéricas escaladas están centradas.

**En `main.ipynb`** este `build_preprocessor()` se montará **dentro de un `Pipeline` de scikit-learn**
junto al clasificador. Así el preprocesado se reajusta en cada fold de la validación cruzada y de
`GridSearchCV`, que es la forma correcta de evitar fugas durante la optimización. No persistimos aquí
el objeto ajustado: se reconstruye desde el código para garantizar reproducibilidad.